# Reemployment example

In [178]:
import pandas as pd
import numpy as np
import patsy
import sys
import statsmodels.formula.api as smf
import statsmodels.api as sm

Choose here between different model specifications and error types. The formula called 'formula' will be used for the following analysis.

In [ ]:
# baseline model
formula_baseline =("(female + black + othrace + C(dep) + q2 + q3 + q4 + q5 + q6 "
           "+ agelt35 + agegt54 + durable + lusd + husd)**2")
# simple model
formula = "(female+black+othrace+C(dep)+agelt35+agegt54+durable+lusd+husd)**2"
# flexible model
formula_flexible= (
    "(female + black + othrace + agelt35 + agegt54 + C(dep) + durable + lusd + husd)**3 "
    "+ (q2 + q3 + q4 + q5 + q6) * (female + black + othrace + agelt35 + agegt54 + C(dep) + durable + lusd + husd)"
    "- agelt35:agegt54 "          # Can't be under 35 AND over 54
    "- husd:lusd "                # Can't be in a high unemployment AND low unemployment site
    "- black:othrace "            # Assuming race categories are mutually exclusive in this dataset
)
error_type = "HC1" # can be changed to "HC0" or "HC1"

See at the end of the code for comments.

## Data

In this lab, we analyze the Pennsylvania re-employment bonus experiment, which was previously studied in "Sequential testing of duration data: the case of the Pennsylvania ‘reemployment bonus’ experiment" (Bilias, 2000), among others. These experiments were conducted in the 1980s by the U.S. Department of Labor to test the incentive effects of alternative compensation schemes for unemployment insurance (UI). In these experiments, UI claimants were randomly assigned either to a control group or one of six treatment groups. Here we focus on treatment group 4, but feel free to explore other treatment groups. In the control group the current rules of UI applied. Individuals in the treatment groups were offered a cash bonus if they found a job within some pre-specified period of time (qualification period), provided that the job was retained for a specified duration. The treatments differed in the level of the bonus, the length of the qualification period, and whether the bonus was declining over time in the qualification period; see http://qed.econ.queensu.ca/jae/2000-v15.6/bilias/readme.b.txt for further details on data.
  

In [222]:
data = pd.read_csv("https://raw.githubusercontent.com/VC2015/DMLonGitHub/master/penn_jae.dat", sep='\\s+')
n, p = data.shape
data = data[data["tg"].isin([0, 4])]

In [223]:
data["T4"] = np.where(data["tg"] == 4, 1, 0)
data["T4"].value_counts()

T4
0    3354
1    1745
Name: count, dtype: int64

In [224]:
data.describe()

,abdt,tg,inuidur1,inuidur2,female,black,hispanic,othrace,dep,q1,...,q6,recall,agelt35,agegt54,durable,nondurable,lusd,husd,muld,T4
count,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,...,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000
mean,10695.416356,1.368896,13.052952,12.281232,0.404001,0.121985,0.032555,0.007256,0.439694,0.012748,...,0.062954,0.110414,0.545009,0.109433,0.148068,0.109237,0.261032,0.218670,0.444205,0.342224
std,111.180503,1.898003,10.565602,10.362143,0.490746,0.327300,0.177487,0.084883,0.757622,0.112194,...,0.242903,0.313436,0.498019,0.312213,0.355202,0.311967,0.439240,0.413385,0.496926,0.474501
min,10404.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10600.000000,0.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,10698.000000,0.000000,11.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,10796.000000,4.000000,25.000000,23.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
max,10880.000000,4.000000,52.000000,52.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Model
To evaluate the impact of the treatments on unemployment duration, we consider the linear regression model:

$$
Y =  D \beta_1 + W'\beta_2 + \varepsilon, \quad E \varepsilon (D,W')' = 0,
$$

where $Y$ is  the  log of duration of unemployment, $D$ is a treatment  indicator,  and $W$ is a set of controls including age group dummies, gender, race, number of dependents, quarter of the experiment, location within the state, existence of recall expectations, and type of occupation.   Here $\beta_1$ is the ATE, assuming the RCT assumptions hold. Within an RCT, the projection coefficient $\beta_1$ has
the interpretation of the causal effect of the treatment on
the average outcome. We thus refer to $\beta_1$ as the average
treatment effect (ATE). Note that the covariates, here are
independent of the treatment $D$, so we can identify $\beta_1$ by
just linear regression of $Y$ on $D$, without adding covariates.
However, we do add covariates in an effort to improve the
precision of our estimates of the average treatment effect.


We also consider an interactive regression model:

$$
Y =  D \alpha_1 + D W' \alpha_2 + W'\beta_2 + \varepsilon, \quad E \varepsilon (D,W', DW')' = 0,
$$
where $W$'s are demeaned (apart from the intercept), so that $\alpha_1$ is the ATE, assuming the RCT assumptions hold. Unlike the model without interactions, we are guaranteed to improve precision by considering the interactive model.

### Analysis

We consider

*  classical 2-sample approach, no adjustment (CL)
*  classical linear regression adjustment (CRA)
*  interactive regression adjusment (IRA)

# Carry out covariate balance check


We first look at the coefficients individually with a $t$-test, and then we adjust the $p$-values to control for family-wise error.

The regression below is done using "type='HC1'" which computes the correct Eicker-Huber-White standard errors, instead of the classical non-robust formula based on homoscedasticity.

In [225]:
"0 ~ " + formula

'0 ~ (female+black+othrace+C(dep)+agelt35+agegt54+durable+lusd+husd)**2'

In [226]:
X = patsy.dmatrix("0 ~ " + formula, data=data, return_type='dataframe')
X.shape

(5099, 55)

There is strong multicollinearity in the design matrix. In order to address this, we drop columns by thresholding using the QR decomposition.

In [227]:
def qr_decomposition(x):
    # Get a set of columns with no linear dependence for smallest subset of observations
    Q, Rx = np.linalg.qr(x)
    ex = np.abs(np.diag(Rx))
    keep = np.where(ex > 1e-6)[0]
    xS = x.iloc[:, keep]

    return xS


xS = qr_decomposition(X)
xS.shape

(5099, 52)

In [228]:
# individual t-tests
covariate_balance = sm.OLS(data["T4"], xS)
cb_fit = covariate_balance.fit(cov_type="HC1", use_t=True)
cb_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     T4   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     53.28
Date:                Fri, 20 Feb 2026   Prob (F-statistic):               0.00
Time:                        21:28:39   Log-Likelihood:                -3396.2
No. Observations:                5099   AIC:                             6896.
Df Residuals:                    5047   BIC:                             7236.
Df Model:                          51                                         
Covariance Type:                  HC1                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.3391      0.026     13.283      0.000       0.289       0.389
C(dep)[T.1]             0.0422      0.050      0.845      0.398      -0.056       0.140
C(dep)[T.2]            -0.0149      0.037     -0.398      0.691      -0.088       0.058
female                 -0.0665      0.030     -2.212      0.027      -0.125      -0.008
female:C(dep)[T.1]      0.0515      0.046      1.118      0.264      -0.039       0.142
female:C(dep)[T.2]      0.0423      0.040      1.048      0.295      -0.037       0.121
black                  -0.0687      0.043     -1.599      0.110      -0.153       0.016
black:C(dep)[T.1]      -0.1246      0.069     -1.809      0.070      -0.260       0.010
black:C(dep)[T.2]      -0.0150      0.063     -0.237      0.813      -0.139       0.109
othrace                 0.1229      0.190      0.646      0.518      -0.250       0.496
othrace:C(dep)[T.1]    -0.6135      0.273     -2.246      0.025      -1.149      -0.078
othrace:C(dep)[T.2]     0.2113      0.166      1.271      0.204      -0.115       0.537
agelt35                -0.0001      0.028     -0.004      0.997      -0.055       0.055
C(dep)[T.1]:agelt35    -0.0733      0.049     -1.508      0.132      -0.169       0.022
C(dep)[T.2]:agelt35    -0.0193      0.038     -0.513      0.608      -0.093       0.054
agegt54                 0.0579      0.051      1.139      0.255      -0.042       0.158
C(dep)[T.1]:agegt54    -0.0675      0.068     -1.000      0.317      -0.200       0.065
C(dep)[T.2]:agegt54     0.0249      0.130      0.191      0.848      -0.230       0.280
durable                -0.0294      0.042     -0.694      0.488      -0.112       0.054
C(dep)[T.1]:durable    -0.0587      0.059     -0.994      0.320      -0.175       0.057
C(dep)[T.2]:durable     0.1215      0.050      2.416      0.016       0.023       0.220
lusd                    0.0376      0.035      1.086      0.277      -0.030       0.105
C(dep)[T.1]:lusd        0.0550      0.052      1.055      0.292      -0.047       0.157
C(dep)[T.2]:lusd        0.0572      0.047      1.212      0.226      -0.035       0.150
husd                    0.0061      0.040      0.152      0.879      -0.073       0.085
C(dep)[T.1]:husd       -0.0559      0.054     -1.027      0.305      -0.163       0.051
C(dep)[T.2]:husd       -0.0906      0.047     -1.944      0.052      -0.182       0.001
female:black            0.0858      0.044      1.967      0.049       0.000       0.171
female:othrace         -0.2089      0.174     -1.199      0.231      -0.550       0.133
female:agelt35          0.0757      0.030      2.522      0.012       0.017       0.135
female:agegt54          0.0251      0.051      0.496      0.620      -0.074       0.124
female:durable          0.0157      0.044      0.360      0.719      -0.070       0.101
fem

To test balance conditions, we employ the Holm-Bonferroni step-down method. With 100+ hypotheses, the family-wise type I error, or the probability of making at least one type I error treating all hypotheses independently, is close to 1. To control for this, we adjust p-values with the following procedure.

First, set $\alpha=0.05$ and denote the list of $n$ p-values from the regression with the vector $p$.

1. Sort $p$ from smallest to largest, so $p_{(1)} \leq p_{(2)} \leq \cdots \leq p_{(n)}$. Denote the corresponding hypothesis for $p_{(i)}$ as $H_{(i)}$.
2. For $i=1,\ldots, n$,
- If $$p_{(i)} > \frac{\alpha}{n-i+1} $$ Break the loop and do not reject any $H_{(j)}$ for $j \geq i$.
- Else reject $H_{(i)}$ if $$p_{(i)} \leq \frac{\alpha}{n-i+1} $$ Increment $i := i+1$.




In [229]:
def holm_bonferroni(p, alpha=0.05):

    n = len(p)
    sig_beta = []

    for i in range(n):
        if np.sort(p)[i] > alpha / (n - i):
            break
        else:
            sig_beta.append(np.argsort(p).iloc[i])
            i += 1

    return sig_beta

index = holm_bonferroni(cb_fit.pvalues, alpha=0.05)
print("Significant Coefficients (Indices): ", index, "\nSignificant Coefficients (Names): ", xS.columns[index])

Significant Coefficients (Indices):  [np.int64(0)] 
Significant Coefficients (Names):  Index(['Intercept'], dtype='object')


We see that that even though this is a randomized experiment, balance conditions are failed.
<!--
The holm method fails to reject any hypothesis. That is, we fail to reject the hypothesis that any coefficient is zero. Thus, in this randomized experiment, balance conditions are met. -->

# Model Specification

In [230]:
# no adjustment (2-sample approach)
cl = smf.ols("np.log(inuidur1) ~ T4", data)

# adding controls
cra = smf.ols("np.log(inuidur1) ~ T4 + " +
              formula,
              data)
# Omitted dummies: q1, nondurable, muld

cl_results = cl.fit(cov_type=error_type)
cl_est = cl_results.params['T4']
if error_type == "HC0":
    cl_se = cl_results.HC0_se['T4']
elif error_type == "HC1":
    cl_se = cl_results.HC1_se['T4']
print(f"ATE: {cl_est:.4f}, (std={cl_se:.4f})")

ATE: -0.0855, (std=0.0359)


In [231]:
cra_results = cra.fit(cov_type=error_type)
cra_est = cra_results.params['T4']
if error_type == "HC0":
    cra_se = cra_results.HC0_se['T4']
elif error_type == "HC1":
    cra_se = cra_results.HC1_se['T4']
print(f"ATE: {cra_est:.4f}, (std={cra_se:.4f})")

ATE: -0.0761, (std=0.0354)


The interactive specification corresponds to the approach introduced in Lin (2013).

In [232]:
# interactive regression model

ira_formula = "0 + (female+black+othrace+C(dep)+q2+q3+q4+q5+q6+agelt35+agegt54+durable+lusd+husd)**2"
X = patsy.dmatrix(ira_formula, data, return_type='dataframe')
X.columns = [f'x{t}' for t in range(X.shape[1])]  # clean column names
X = (X - X.mean(axis=0))  # demean all control covariates

# construct interactions of treatment and (de-meaned covariates, 1)
ira_formula = "T4 * (" + "+".join(X.columns) + ")"
X['T4'] = data['T4']
X = patsy.dmatrix(ira_formula, X, return_type='dataframe')
ira = sm.OLS(np.log(data[["inuidur1"]]), X)
ira_results = ira.fit(cov_type="HC0")
ira_est = ira_results.params['T4']

# because we de-meaned the covariates and interacted them with the treamtent
# we need to account for the error in the means of X, which has a first-order
# effect on the ATE estimate
interaction_cols = list(map(lambda x: x.startswith('T4:'), list(X.columns)))
cate = X.iloc[:, interaction_cols] @ ira_results.params[interaction_cols]
ira_se = np.sqrt(ira_results.HC0_se['T4']**2 + np.var(cate) / X.shape[0])

print(f"ATE: {ira_est:.4f}, (std={ira_se:.4f})")

ATE: -195221331.3866, (std=244527348.3408)


In [233]:
print("Python version:", sys.version)
print("patsy version:", patsy.__version__)

Python version: 3.13.7 (main, Aug 14 2025, 11:12:11) [Clang 17.0.0 (clang-1700.0.13.3)]
patsy version: 1.0.2


For some reason my local version of python returns a completely wird result for IRA. Maybe it is because my local python version is 3.13.7 while on collab it is 3.12.12.

# Partialling Out with the Lasso
Next we try out partialling out with theoretically justified lasso. We use "plug-in" tuning with a theoretically valid choice of penalty $\lambda = 2 \cdot c \hat{\sigma} \sqrt{n} \Phi^{-1}(1-\alpha/2p)$, where $c>1$ and $1-\alpha$ is a confidence level, and $\Phi^{-1}$ denotes the quantile function. See book for more details.

We pull an analogue of R's rlasso. We find a Python code that tries to replicate the main function of hdm r-package. It was made by [Max Huppertz](https://maxhuppertz.github.io/code/). His library is this [repository](https://github.com/maxhuppertz/hdmpy). If not using colab, download its repository and copy this folder to your site-packages folder. In my case it is located here ***C:\Python\Python38\Lib\site-packages*** . We need to install this package ***pip install multiprocess***.

In [234]:
#!pip install multiprocess
#!pip install pyreadr
#!git clone https://github.com/maxhuppertz/hdmpy.git

In [235]:
import sys
sys.path.insert(1, "./hdmpy")

In [236]:
# We wrap the package so that it has the familiar sklearn API
import hdmpy
from sklearn.base import BaseEstimator


class RLasso(BaseEstimator):

    def __init__(self, *, post=True):
        self.post = post

    def fit(self, X, y):
        self.rlasso_ = hdmpy.rlasso(X, y, post=self.post)
        return self

    def predict(self, X):
        return np.array(X) @ np.array(self.rlasso_.est['beta']).flatten() + np.array(self.rlasso_.est['intercept'])

    @property
    def coef_(self,):
        return np.array(self.rlasso_.est['beta']).flatten()


def lasso_model():
    return RLasso(post=False)

## Non-Interactive Model

In [237]:
y = np.log(data["inuidur1"])
D = data['T4']
ira_formula = "0 + " + formula
X = patsy.dmatrix(ira_formula, data, return_type='dataframe')
X = X.loc[:, X.std() > 0]  # drop constant columns

Dres = D - np.mean(D)
yres = y - lasso_model().fit(X, y).predict(X)

cra_lasso_results = sm.OLS(yres, Dres).fit(cov_type=error_type)
cra_lasso_est = cra_lasso_results.params['T4']
if error_type == "HC0":
    cra_lasso_se = cra_lasso_results.HC0_se['T4']
elif error_type == "HC1":
    cra_lasso_se = cra_lasso_results.HC1_se['T4']
print(f"ATE: {cra_lasso_est:.4f}, (std={cra_lasso_se:.4f})")

ATE: -0.0793, (std=0.0354)


## Interactive Model

In [238]:
X.columns = [f'x{t}' for t in range(X.shape[1])]  # clean column names
X = (X - X.mean(axis=0))  # demean all control covariates

# construct interactions of treatment and (de-meaned covariates, 1)
ira_formula = "T4 * (" + "+".join(X.columns) + ")"
X['T4'] = data['T4']
X = patsy.dmatrix(ira_formula, X, return_type='dataframe')
# drop treatment column
X = X.drop('T4', axis=1)
X = X.loc[:, X.std() > 0]  # drop constant columns

Dres = D - np.mean(D)
modely = lasso_model().fit(X, y)
yres = y - modely.predict(X)


ira_lasso_results = sm.OLS(yres, Dres).fit(cov_type=error_type)
ira_lasso_est = ira_lasso_results.params['T4']

# because we de-meaned the covariates and interacted them with the treatment
# we need to account for the error in the means of X, which has a first-order
# effect on the ATE estimate
interaction_cols = list(map(lambda x: x.startswith('T4:'), list(X.columns)))
cate = X.iloc[:, interaction_cols] @ modely.coef_[interaction_cols]
if error_type == "HC0":
    ira_lasso_se = np.sqrt(ira_lasso_results.HC0_se['T4']**2 + np.var(cate) / X.shape[0])
elif error_type == "HC1":
    ira_lasso_se = np.sqrt(ira_lasso_results.HC1_se['T4']**2 + np.var(cate) / X.shape[0])

# Overall Results

In [239]:
results = {
    "CL": [cl_est, cl_se],
    "CRA": [cra_est, cra_se],
    "IRA": [ira_est, ira_se],
    "CRA w Lasso": [cra_lasso_est, cra_lasso_se],
    "IRA w Lasso": [ira_lasso_est, ira_lasso_se],
}
results = pd.DataFrame(results)
results.index = ["Estimate", "Standard Error"]
results
# for printing to LaTeX: print(results.to_latex())

,CL,CRA,IRA,CRA w Lasso,IRA w Lasso
Estimate,-0.085455,-0.076110,-1.952213e+08,-0.079349,-0.068311
Standard Error,0.035856,0.035431,2.445273e+08,0.035448,0.035368


Treatment group 4 experiences an average decrease of about $7.8\%$ in the length of unemployment spell.


Observe that regression estimators delivers estimates that are slighly more efficient (lower standard errors) than the simple 2 mean estimator, but essentially all methods have very similar standard errors.  We also see the regression estimators offer slightly lower estimates -- these difference occur perhaps to due minor imbalance in the treatment allocation, which the regression estimators try to correct.




In [240]:
#results_H0 = results.loc["Standard Error"]
#results_H0

CL             3.585569e-02
CRA            3.543070e-02
IRA            2.445273e+08
CRA w Lasso    3.544780e-02
IRA w Lasso    3.536765e-02
Name: Standard Error, dtype: float64

In [220]:
#results_H1 = results.loc["Standard Error"]
#results_H1

CL             3.585569e-02
CRA            3.622005e-02
IRA            2.445273e+08
CRA w Lasso    3.537229e-02
IRA w Lasso    3.529340e-02
Name: Standard Error, dtype: float64

In [241]:
#print((results_H1 - results_H0))

CL             0.000000
CRA            0.000789
IRA            0.000000
CRA w Lasso   -0.000076
IRA w Lasso   -0.000074
Name: Standard Error, dtype: float64


For some reason my local version of python returns a completely wird result for IRA. Maybe it is because my local python version is 3.13.7 while on collab it is 3.12.12. Since the IRA model with Lasso works, I did not spend much more time on trying to get the code running on my machine.

## Standard Error: Difference HC1 - HC0 
in simple/flexible control configuration

| Model | Value |
| :--- | :--- |
| CL | 0.000007 |
| CRA | 0.000185 / 0.000638 |
| CRA w Lasso | 0.000003 |
| IRA w Lasso | 0.000003 |

As expected robust HC1 standard errors are larger than standard HC0 standard errors, which uses asymptotics. Since the finite sample has 5,099 observations the difference is not too large. However, when a more complex model is used, the difference increases as correcting for the degrees of freedom becomes more important and the errors noticably larger (only really visible for CRA).

Although we would expect adding controls to help the precision of our RCT ATE, this is not the case when we keep too many unrelevant interactions in our regression under robust standard erorrs, as this inflates the dof correction.

## Difference flexible model - simple model
### ATE
| Model | Value |
| :--- | :--- |
| CL | no model difference |
| CRA | -0.010772 |
| CRA w Lasso | -0.002492 |
| IRA w Lasso | -0.005110 |

In an RCT, adding covariates is not necessary for identification. The ATE should remain relatively stable if the RCT was successful, as treatment should independent of W. This is what we can roughly observe. The CRA estimate shifts the most because it is probably suffering from mild overfitting.

### Standard Error (HC1)
| Model | Value |
| :--- | :--- |
| CL | no model difference |
| CRA | 0.000789 |
| CRA w Lasso | -0.000076 |
| IRA w Lasso | -0.000074 |


We would expect in an RCT that adding covariates is beneficial for precision.

CRA: This illustrates the curse of dimensionality in unpenalized OLS. When adding many (almost) irrelevant interaction terms, the number of parameters increases massively. Using HC1 standard errors (which correct for degrees of freedom​), this massive number of regressors inflates the standard error.

Penalized regressions: This visualizes why we use Lasso: to perform variable selection. Lasso picks from the many interactions the few that are highly predictive and shrinks the rest to zero. Because the Lasso found better, highly predictive non-linear terms (that weren't available in the simple model), it explained more of the residual variance of the outcome. Lower residual variance translates directly to tighter, smaller standard errors, all without suffering the HC1-penalty (robust SE) of standard OLS for many more regressors.


Thus, for our data adding numerous highly flexible controls improves precision, but only if using a regularization method like Lasso to manage the dimensionality. If you just throw everything into a standard OLS (CRA), you actually lose precision and hurt your standard errors.

Additionally, including more controls in the flexible model made the covariates more balanced in the initial check.